<a href="https://colab.research.google.com/github/AhmedEssam1996/AhmedEssam1996/blob/main/Copy_of_rag_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  **RAG System With Lang Chain**

In [ ]:
!pip install -qU langchain langchain-community langchain-core langchain-huggingface
!pip install -qU chromadb pypdf sentence-transformers gpt4all

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [ ]:
import torch
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


In [ ]:
# 1. إعداد الملف والبيانات
file_path = "/content/Supervised-Machine-Learning.pdf"

if not os.path.exists(file_path):
    print("❌ ارفع ملف الـ PDF أولاً باسم 'Supervised-Machine-Learning.pdf'")
else:
    loader = PyPDFLoader(file_path)
    pages = loader.load()

    text_splitter = CharacterTextSplitter(separator="\n", chunk_size=1000, chunk_overlap=100)
    docs = text_splitter.split_documents(pages)

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:

# Prompt
template = """You are a helpful and accurate assistant.
Answer the following question strictly based on the context provided.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

# Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)

In [ ]:
# التجربة
query = "What is The regression problem?"
print(f"\n🔍 Searching for: {query}")

response = qa_chain.invoke({"query": query})

print("\n💡 Response:\n", response["result"])


🔍 Searching for: What is The regression problem?


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



💡 Response:
 You are a helpful and accurate assistant. 
Answer the following question strictly based on the context provided.
If the answer is not in the context, say "I don't know."

Context:
2 The regression problem and linear regression
The ﬁrst problem we will study is theregression problem. Regression is one of the two main problems
that we cover in these notes. (The other one is classiﬁcation). The ﬁrst method we will encounter islinear
regression,whichisone(ofmany)solutionstotheregressionproblem. Eventhoughtherelativesimplicity
of linear regression, it is surprisingly useful and it also constitutes an important building block for more
advanced methods (such as deep learning, Chapter 7).
2.1 The regression problem
Regression refers to the problem of learning the relationships between some (qualitative or quantitative1)
inputvariables x = [x1x2 ... xp]Tandaquantitativeoutputvariable y. Inmathematicalterms,regression
is about learning a modelf
y =f(x) +ε, (2.1)
whereε is a noise/e

________________________________________________________________________________
________________________________________________________________________________

# **Rag System With LLama Index**

In [ ]:
# اجمعهم كلهم هنا
!pip install -qU llama-index llama-index-llms-huggingface llama-index-embeddings-huggingface transformers accelerate pypdf bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 8.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
!pip install -qU llama-index-readers-file pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.1 MB/s eta 0:00:00


In [ ]:
import torch
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, ServiceContext, Settings
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import os



In [ ]:
from llama_index.readers.file import PDFReader
from pathlib import Path

# 1. تحديد مسار الملف بدقة
file_path = "/content/data/Supervised-Machine-Learning.pdf"

# 2. استخدام القارئ المتخصص للـ PDF
loader = PDFReader()
documents = loader.load_data(file=Path(file_path))

# 3. التحقق من النتيجة
print(f"✅ تم تحميل المستند بنجاح!")
print(f"📄 عدد الصفحات الفعلي: {len(documents)}")

# معاينة للتأكد من وجود نص
if len(documents) > 0:
    print(f"🔍 بداية النص في الصفحة الأولى:\n {documents[0].text[:200]}...")

✅ تم تحميل المستند بنجاح!
📄 عدد الصفحات الفعلي: 112
🔍 بداية النص في الصفحة الأولى:
 Supervised Machine Learning
Lecture notes for the Statistical Machine Learning course
Andreas Lindholm, Niklas Wahlström,
Fredrik Lindsten, Thomas B. Schön
Version: March 12, 2019
Department of Inform...


In [ ]:
    # ==========================================
    # 2. إعداد الـ Embeddings والموديل (Qwen)
    # ==========================================

    # إعداد موديل الـ Embeddings
    Settings.embed_model = HuggingFaceEmbedding(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    # إعداد موديل Qwen2.5-1.5B
    model_id = "Qwen/Qwen2.5-1.5B-Instruct"

    Settings.llm = HuggingFaceLLM(
        model_name=model_id,
        tokenizer_name=model_id,
        context_window=3900,
        max_new_tokens=512,
        generate_kwargs={"temperature": 0.1, "do_sample": True},
        device_map="auto",
        model_kwargs={"torch_dtype": torch.float16} # لتسريع الأداء وتقليل الذاكرة
    )



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
    # ==========================================
    # 3. بناء الفهرس (Indexing) ومحرك البحث
    # ==========================================

    # تحويل المستندات إلى Vector Index
    index = VectorStoreIndex.from_documents(documents)

    # تخصيص الـ Prompt ليناسب طلبك
    template = (
        "Context information is below.\n"
        "---------------------\n"
        "{context_str}\n"
        "---------------------\n"
        "Given the context information and not prior knowledge, "
        "answer the query concisely and clearly. Use the same terminology as the document.\n"
        "If the answer is not in the context, say 'I don't know.'\n"
        "Query: {query_str}\n"
        "Answer: "
    )
    qa_prompt = PromptTemplate(template)

    # إنشاء محرك الاستعلام (Query Engine)
    query_engine = index.as_query_engine(
        text_qa_template=qa_prompt,
        similarity_top_k=3 # استرجاع أفضل 3 قطع نصية
    )



In [ ]:
    # ==========================================
    # 4. التجربة
    # ==========================================
    query = "What is The regression problem?"
    print(f"\n🔍 Querying LlamaIndex: {query}")

    response = query_engine.query(query)
    print("\n💡 Response:\n", response)


🔍 Querying LlamaIndex: What is The regression problem?

💡 Response:
 2 The regression problem and linear regression
The first problem we will study is theregression problem. Regression is one of the two main problems
that we cover in these notes. (The other one is classiﬁcation). The ﬁrst method we will encounter islinear
regression,whichisone(ofmany)solutionstotheregressionproblem. Eventhoughtherelativesimplicity
of linear regression, it is surprisingly useful and it also constitutes an important building block for more
advanced methods (such as deep learning, Chapter 7).


In [ ]:
!pip install nbstripout
!nbstripout rag_system.ipynb

Could not strip 'rag_system.ipynb': file not found
